In [1]:
import pandas as pd
import numpy as np
import scanpy as sc

Download datasets from [CellNavi GitHub](https://github.com/DLS5-Omics/CellNavi), which provides example dataset in [DropBox](https://www.dropbox.com/scl/fo/rq9klah7vqksn6e66dsae/AK3DJ2sxwL3MoWCOcQ9ZfFE?rlkey=1t4kz2vraif0ifu72c6gmo6xl&e=1&st=gpvwfw3j&dl=0). We merge Re-stimulated_t_example_train.h5ad and Resting_t_example_test.h5ad together to form a single dataset and preprocess it.

In [2]:
adata = sc.concat({
    'train': sc.read('Re-stimulated_t_example_train.h5ad'),
    'test': sc.read('Resting_t_example_test.h5ad'),
}, label='split')
adata.obs['perturbation'] = adata.obs['perturbation'].replace({'NO-TARGET': 'control'})
adata.layers['raw'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(
    adata, n_top_genes=5000, 
    subset=False, flavor='seurat',
)
print(adata.shape)

/tmp/ipykernel_2360752/3260245825.py:5: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  adata.obs['perturbation'] = adata.obs['perturbation'].replace({'NO-TARGET': 'control'})


(8848, 19240)


In [3]:
adata.obs[['split', 'condition']].value_counts()

split  condition    
train  Re-stimulated    5690
test   Resting          3158
Name: count, dtype: int64

In [3]:
adata.write_h5ad('preprocessed.h5ad', compression='gzip')

## split dataset

In [ ]:
adata = sc.read_h5ad('preprocessed.h5ad')

In [4]:
np.random.seed(42)
split_df = (
    adata
    .obs[['condition']]
    .copy()
    .reset_index(names='cell')
    .rename(columns={'condition': 'subsplit'})
)
split_df['presplit'] = np.random.choice(
    ['train', 'val', 'test'], 
    size=split_df.shape[0], 
    p=[0.7, 0.1, 0.2], 
    replace=True
)


In [5]:
# train on Re-stimulated and test on Resting
split_df['split'] = split_df[['subsplit', 'presplit']].apply(
    lambda x: x['presplit'] if x['subsplit'] == 'Re-stimulated' else 'test', axis=1
)
split_df[['cell', 'split', 'subsplit']].to_csv('split_trainonstimulated.csv', index=False)
pd.read_csv('split_trainonstimulated.csv').groupby(['split', 'subsplit']).size()


split  subsplit     
test   Re-stimulated    1132
       Resting          3158
train  Re-stimulated    4022
val    Re-stimulated     536
dtype: int64

In [6]:
# train on Resting and test on Re-stimulated=
split_df['split'] = split_df[['subsplit', 'presplit']].apply(
    lambda x: x['presplit'] if x['subsplit'] == 'Resting' else 'test', axis=1
)
split_df[['cell', 'split', 'subsplit']].to_csv('split_trainonresting.csv', index=False)
pd.read_csv('split_trainonresting.csv').groupby(['split', 'subsplit']).size()

split  subsplit     
test   Re-stimulated    5690
       Resting           613
train  Resting          2271
val    Resting           274
dtype: int64